## Model Serving - Solution

Local inference server using MLflow for batch and online inference on the Churn dataset.

In [17]:
import os
os.chdir(r'c:\Users\ancas\Downloads\Session 3 - Actitivies-20260526\Class Exercise')
print(f'Working directory: {os.getcwd()}')

Working directory: c:\Users\ancas\Downloads\Session 3 - Actitivies-20260526\Class Exercise


### Setup

Start MLflow server before running: `mlflow server --host 127.0.0.1 --port 8080`

In [18]:
import pandas as pd
import mlflow
import joblib
import requests
import json
from exercise_support import ClassSuportTransformer, Model, MLFlowRunExecution

### Model Training

In [19]:
# Load and transform data
df = pd.read_csv('Churn_Modelling_train_test.csv')
transformer = ClassSuportTransformer()
df_transformed = transformer.transform_class_support(df)
df_transformed = transformer.balance_dataset(df_transformed)
df_transformed.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Geography_nan
0,674,0,45.0,7,142072.02,1,1.0,0.0,37013.29,0,0.0,0.0,0.0
1,438,1,54.0,2,0.00,1,0.0,0.0,191763.07,1,0.0,1.0,0.0
2,749,0,47.0,9,110022.74,1,0.0,1.0,135655.29,1,1.0,0.0,0.0
3,724,0,34.0,6,118235.70,2,0.0,0.0,157137.23,0,1.0,0.0,0.0
4,586,1,46.0,0,0.00,3,0.0,1.0,131553.82,1,0.0,0.0,0.0


In [20]:
# Save encoder
PATH = './'
encoder = joblib.load('encoder.pkl')
joblib.dump(encoder, f'{PATH}one_hot_encoder.pkl')
print('Encoder saved to one_hot_encoder.pkl')

Encoder saved to one_hot_encoder.pkl


In [21]:
# Train and register model in MLflow
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

model_trainer = Model()
mlflow_input = model_trainer.train_model(df_transformed)

mlflow_runner = MLFlowRunExecution()
mlflow_runner.run_mlflow_execution(mlflow_input)

print(f'Model accuracy: {mlflow_input.metric:.4f}')

c:\Users\ancas\Downloads\Session 3 - Actitivies-20260526\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/01 18:46:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/01 18:46:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle 

🏃 View run ambitious-smelt-242 at: http://127.0.0.1:8080/#/experiments/1/runs/edf802ef83494190a8776db55e828697
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1
Model accuracy: 0.7337


Created version '20' of model 'class-churn-model'.


### Inference

In [22]:
# Load and transform validation data
df_validation = pd.read_csv('Churn_Modelling_val.csv')
transformer_val = ClassSuportTransformer()
df_val_transformed = transformer_val.transform_class_support(df_validation)
X_validation = df_val_transformed.drop('Exited', axis=1)
y_validation = df_val_transformed['Exited']
print(f'Validation set shape: {X_validation.shape}')

Validation set shape: (1001, 12)


##### Batch Inference

In [23]:
# define a function to implement batch inference with mlflow
def batch_inference(model_uri: str, input: pd.DataFrame):
    model = mlflow.pyfunc.load_model(model_uri)
    predictions = model.predict(input)
    return predictions

In [24]:
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")
model_uri = "models:/class-churn-model/latest"

# Load model and store for reuse in online inference
_model = mlflow.pyfunc.load_model(model_uri)

batch_prediction_result = _model.predict(X_validation)
print(f'Predictions: {batch_prediction_result[:10]}')

Predictions: [0 0 1 0 0 0 0 0 0 1]


In [25]:
from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_validation, batch_prediction_result)
print('Confusion Matrix:')
print(cm)
print()
print(classification_report(y_validation, batch_prediction_result))

Confusion Matrix:
[[579 224]
 [ 62 136]]

              precision    recall  f1-score   support

           0       0.90      0.72      0.80       803
           1       0.38      0.69      0.49       198

    accuracy                           0.71      1001
   macro avg       0.64      0.70      0.64      1001
weighted avg       0.80      0.71      0.74      1001



##### Online Inference

In [26]:
# Load one record for online inference
df_online = pd.read_csv('Churn_Modelling_val.csv').head(1)
transformer_online = ClassSuportTransformer()
df_online_transformed = transformer_online.transform_class_support(df_online)
X_online = df_online_transformed.drop('Exited', axis=1)
print('Input record:')
X_online

Input record:


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Geography_nan
0,807,0,42.0,5,0.0,2,1.0,1.0,74900.9,0.0,1.0,0.0


In [27]:
def get_inference_endpoint(host="http://127.0.0.1", port=5000):
    return f"{host}:{port}/invocations"

url = get_inference_endpoint()
print(f'Inference endpoint: {url}')

Inference endpoint: http://127.0.0.1:5000/invocations


In [28]:
# define a function to implement online inference with mlflow - pandas input
def online_inference_pandas(url: str, input: pd.DataFrame):
    payload = {"dataframe_split": input.to_dict(orient='split')}
    headers = {"Content-Type": "application/json"}
    response = requests.post(url, json=payload, headers=headers)
    return response

In [29]:
response_pandas = online_inference_pandas(url, X_online)
print(f'Status code: {response_pandas.status_code}')
response_pandas.content

Status code: 500


b"Failed to enforce schema of data '   CreditScore  Gender  Age  Tenure  Balance  ...  IsActiveMember  EstimatedSalary  Geography_Germany  Geography_Spain  Geography_nan\n0          807       0   42       5        0  ...               1          74900.9                  0                1              0\n\n[1 rows x 12 columns]' with schema '['CreditScore': long (required), 'Gender': long (required), 'Age': double (required), 'Tenure': long (required), 'Balance': double (required), 'NumOfProducts': long (required), 'HasCrCard': double (required), 'IsActiveMember': double (required), 'EstimatedSalary': double (required), 'Geography_Germany': double (required), 'Geography_Spain': double (required), 'Geography_nan': double (required)]'. Error: Incompatible input types for column Age. Can not safely convert int64 to float64."

In [30]:
# define a function to implement online inference with mlflow - json input
def online_inference_json(url: str, input: dict):
    headers = {"Content-Type": "application/json"}
    response = requests.post(url, json=input, headers=headers)
    return response

In [31]:
# define the json as required by MLFlow
input_json = {
    "dataframe_split": {
        "columns": X_online.columns.tolist(),
        "data": X_online.values.tolist()
    }
}
print(json.dumps(input_json, indent=2))

{
  "dataframe_split": {
    "columns": [
      "CreditScore",
      "Gender",
      "Age",
      "Tenure",
      "Balance",
      "NumOfProducts",
      "HasCrCard",
      "IsActiveMember",
      "EstimatedSalary",
      "Geography_Germany",
      "Geography_Spain",
      "Geography_nan"
    ],
    "data": [
      [
        807.0,
        0.0,
        42.0,
        5.0,
        0.0,
        2.0,
        1.0,
        1.0,
        74900.9,
        0.0,
        1.0,
        0.0
      ]
    ]
  }
}


In [32]:
response_json = online_inference_json(url, input_json)
print(f'Status code: {response_json.status_code}')
response_json.content

Status code: 500


b"Failed to enforce schema of data '   CreditScore  Gender  Age  Tenure  Balance  ...  IsActiveMember  EstimatedSalary  Geography_Germany  Geography_Spain  Geography_nan\n0          807       0   42       5        0  ...               1          74900.9                  0                1              0\n\n[1 rows x 12 columns]' with schema '['CreditScore': long (required), 'Gender': long (required), 'Age': double (required), 'Tenure': long (required), 'Balance': double (required), 'NumOfProducts': long (required), 'HasCrCard': double (required), 'IsActiveMember': double (required), 'EstimatedSalary': double (required), 'Geography_Germany': double (required), 'Geography_Spain': double (required), 'Geography_nan': double (required)]'. Error: Incompatible input types for column Age. Can not safely convert int64 to float64."